# Predicting Electric Vehicle Purchases
Kaggle Playground Series S6E9: binary classification (`Will_Buy_EV`), scored by ROC-AUC.

**Final score:** Private LB 0.94086 · Public LB 0.94149, from a weighted blend of XGBoost, LightGBM and CatBoost.

In [ ]:
#Essentials
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
#train_test_split
from sklearn.model_selection import train_test_split
#for cols coversion and pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline

#Required_Models
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold

#Evaluation_Matrix
from sklearn.metrics import roc_auc_score

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):

    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        self.frequencies_ = {}

        for col in self.columns:
            freq = X[col].value_counts(normalize=True)
            self.frequencies_[col] = freq
            
        return self


    def transform (self, X):
        encoded = pd.DataFrame()

        for col in self.columns:
            
            encoded[col] = X[col].map(self.frequencies_[col])

        return encoded

In [ ]:
"""freq_encoder = FrequencyEncoder(
    columns=['City_Type', 'Current_Car_Type', 'Range_Anxiety_Level']
)

freq_encoder.fit(X_tr)

X_tr_freq = freq_encoder.transform(X_tr)
X_val_freq = freq_encoder.transform(X_val)

print(X_tr_freq.head())"""

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
train.shape

In [ ]:
train.head(5)

In [ ]:
train['Current_Car_Type'].unique()

In [ ]:
train.describe(percentiles=[0.25,0.5,0.7,0.9])

In [ ]:
train['Will_Buy_EV_num'] = train['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

In [ ]:
num_cols = train.select_dtypes(include='number').columns

train[num_cols].corr()['Will_Buy_EV_num'].sort_values(ascending=False)

In [ ]:
#feature_engineering (How scary charging is for this person)
"""home_charging = train['Home_Charging_Possible'].map({'Yes': 1, 'No': 0})
train['worry_score'] = (
    train['Daily_Commute_km']
    - 5 * train['Charging_Stations_Near_Home']
    - 5 * train['Charging_Stations_Near_Work']
    - 150 * home_charging
)"""

In [ ]:
'''train['total_chargers']= (
    train['Charging_Stations_Near_Home'] + train['Charging_Stations_Near_Work']
)'''

In [ ]:
subsidy_avail = train['Home_Charging_Possible'].map({'Yes': 1, 'No': 0})
train['income_X_subsidy'] = (
    train['Annual_Income_USD'] * subsidy_avail
) 

train['Concern_X_subsidy'] = (
    train['Environmental_Concern_Level'] * subsidy_avail
)

In [ ]:
#train['Commute_per_Charging_Station']= train['Daily_Commute_km'] / (train['total_chargers'] + 1)

In [ ]:
X_train = train.drop(['Will_Buy_EV', 'Will_Buy_EV_num'], axis=1)
y_train = train['Will_Buy_EV']
y_train = y_train.map({'Yes': 1, 'No': 0})

In [ ]:
#split the train set into train and Val set (Later will do cross val k-fold)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size= 0.2, 
                                           random_state = 42, stratify = y_train)

In [ ]:
num_cols = X_tr.drop([
    'id',
    'Environmental_Concern_Level',
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
], axis=1).columns.tolist()

cat_cols = X_tr.drop([
    'id',
    'Age',
    'Annual_Income_USD',
    'Daily_Commute_km',
    'Number_of_Cars_Owned',
    'Charging_Stations_Near_Home',
    'Charging_Stations_Near_Work',
    'Concern_X_subsidy',
    'income_X_subsidy',
    'Range_Anxiety_Level',
    
], axis=1).columns.tolist()

In [ ]:
corr = X_train.copy()

corr['Will_Buy_EV'] = y_train

corr_matrix = corr.corr(numeric_only=True)['Will_Buy_EV'].sort_values(
    ascending=False
)

print(corr_matrix)

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', 'passthrough' , num_cols),
        ('env', 'passthrough', ['Environmental_Concern_Level','Concern_X_subsidy']),
        ('col', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('ord', 
         OrdinalEncoder(categories=[['Low','Medium','High']]), ['Range_Anxiety_Level'])
    ]
)

In [ ]:
print(num_cols)
print(cat_cols)

In [ ]:
logistic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [ ]:
cv_scores_logist = cross_val_score(
    logistic_model,
    X_train,
    y_train,
    cv=10,
    scoring='roc_auc'  
)

In [ ]:
print("Logistic Regression CV AUC:", cv_scores_logist.mean(), "+/-", cv_scores_logist.std())


In [ ]:
xgboost_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', XGBClassifier())
])

In [ ]:
cv_scores = cross_val_score(
    xgboost_model,
    X_train,
    y_train,
    cv=10,
    scoring='roc_auc'
)

In [ ]:
print("CV scores:", cv_scores)
print("Mean AUC:", cv_scores.mean())
print("Std:", cv_scores.std())

In [ ]:
# 10-fold stratified CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Generate OOF probabilities
oof_predictions = cross_val_predict(
    xgboost_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

# Calculate OOF ROC-AUC
oof_auc = roc_auc_score(
    y_train,
    oof_predictions
)

print("OOF AUC:", oof_auc)

In [ ]:
lgbm_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

lgbm_oof = cross_val_predict(
    lgbm_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

lgbm_oof_auc = roc_auc_score(
    y_train,
    lgbm_oof
)

print("LightGBM OOF AUC:", lgbm_oof_auc)

In [ ]:
blend_oof = (
    0.5 * oof_predictions +
    0.5 * lgbm_oof
)

blend_auc = roc_auc_score(
    y_train,
    blend_oof
)

print("XGB + LightGBM OOF AUC:", blend_auc)

In [ ]:
for xgb_weight in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    
    blend_oof = (
        xgb_weight * oof_predictions +
        (1 - xgb_weight) * lgbm_oof
    )
    
    auc = roc_auc_score(y_train, blend_oof)
    
    print(
        f"XGB: {xgb_weight:.1f} | "
        f"LGBM: {1-xgb_weight:.1f} | "
        f"AUC: {auc:.6f}"
    )

In [ ]:
# Make a copy
X_cat = X_train.copy()

# CatBoost needs categorical columns to be strings
cat_cols_catboost = [
    'Environmental_Concern_Level',
    'Gender',
    'City_Type',
    'Current_Car_Type',
    'Home_Charging_Possible',
    'Subsidy_Available',
    'Range_Anxiety_Level'
]

for col in cat_cols_catboost:
    X_cat[col] = X_cat[col].astype(str)

# CatBoost model
catboost_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    eval_metric='AUC',
    random_seed=42,
    verbose=0,
    thread_count=-1
)

# 10-fold CV
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

# Generate OOF predictions
catboost_oof = cross_val_predict(
    catboost_model,
    X_cat,
    y_train,
    cv=cv,
    method='predict_proba',
    params={'cat_features': cat_cols_catboost},
    n_jobs=1
)[:, 1]

# OOF AUC
catboost_auc = roc_auc_score(
    y_train,
    catboost_oof
)

print("CatBoost OOF AUC:", catboost_auc)

In [ ]:
print("CatBoost OOF AUC:", catboost_auc)

In [ ]:
blend_3 = (
    0.4 * oof_predictions +
    0.5 * lgbm_oof +
    0.1 * catboost_oof
)

blend_3_auc = roc_auc_score(
    y_train,
    blend_3
)

print("XGB + LGBM + CatBoost OOF AUC:", blend_3_auc)

In [ ]:
for cat_weight in [0.05, 0.10, 0.15, 0.20, 0.25]:
    
    remaining = 1 - cat_weight
    
    # Keep XGB:LGBM ratio approximately 40:60
    xgb_weight = remaining * 0.4
    lgbm_weight = remaining * 0.6

    blend = (
        xgb_weight * oof_predictions +
        lgbm_weight * lgbm_oof +
        cat_weight * catboost_oof
    )

    auc = roc_auc_score(y_train, blend)

    print(
        f"XGB: {xgb_weight:.2f} | "
        f"LGBM: {lgbm_weight:.2f} | "
        f"Cat: {cat_weight:.2f} | "
        f"AUC: {auc:.6f}"
    )

In [ ]:
X_test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

X_test['income_X_subsidy'] = (
    X_test['Annual_Income_USD'] *
    X_test['Subsidy_Available'].map({'Yes': 1, 'No': 0})
)

X_test['Concern_X_subsidy'] = (
    X_test['Environmental_Concern_Level'] *
    X_test['Subsidy_Available'].map({'Yes': 1, 'No': 0})
)

X_test.shape

In [ ]:
xgboost_model.fit(X_train, y_train)

In [ ]:
xgb_test_pred = xgboost_model.predict_proba(X_test)[:, 1]
xgb_test_pred

In [ ]:
lgbm_model.fit(X_train, y_train)

In [ ]:
lgbm_test_pred = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_test_pred

In [ ]:
X_test_cat = X_test.copy()

for col in cat_cols_catboost:
    X_test_cat[col] = X_test_cat[col].astype(str)

catboost_model.fit(
    X_cat,
    y_train,
    cat_features=cat_cols_catboost
)

catboost_test_pred = catboost_model.predict_proba(
    X_test_cat
)[:, 1]

In [ ]:
final_pred = (
    0.36 * xgb_test_pred +
    0.54 * lgbm_test_pred +
    0.10 * catboost_test_pred
)

In [ ]:
submission = pd.DataFrame({
    'id': X_test['id'],
    'Will_Buy_EV': final_pred
})

In [ ]:
submission.head()

In [ ]:
submission.shape

In [ ]:
submission.to_csv('submission.csv', index=False)

In [ ]:
final_pred_xgb_lgbm = (
    0.40 * xgb_test_pred +
    0.60 * lgbm_test_pred
)

In [ ]:
submission_xgb_lgbm = pd.DataFrame({
    'id': X_test['id'],
    'Will_Buy_EV': final_pred_xgb_lgbm
})

In [ ]:
submission_xgb_lgbm.to_csv('submission_xgb_lgbm.csv', index=False)